In [1]:
from utils import State, Action
from collections import OrderedDict
from torch import tensor


In [ ]:
coeffs

In [3]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from utils import ImmutableState, State, Action, board_status, get_all_valid_actions, is_terminal, change_state, terminal_utility, invert, load_data
import time
import numpy as np

class UTTTModel(nn.Module):
    def __init__(self):
        super(UTTTModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        #self.conv4 = nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 9 * 9, 192)  # 16 channels from last conv layer
        self.fc2 = nn.Linear(192, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        #self.prelu = nn.PReLU()
        self.lrelu = nn.LeakyReLU(0.01)
    
    def forward(self, x):
        x = x.view(-1, 3, 9, 9)  # Reshape to 9x9 grid
        x = self.lrelu(self.conv1(x))
        x = self.lrelu(self.conv2(x))
        x = self.lrelu(self.conv3(x))
        #x = self.lrelu(self.conv4(x))
        x = x.view(x.size(0), -1)  # Flatten for FC layers
        x = self.lrelu(self.fc1(x))
        x = self.lrelu(self.fc2(x))
        x = self.lrelu(self.fc3(x))
        #return torch.sigmoid(self.fc4(x)) #output range (0,1)
        return torch.tanh(self.fc4(x))  # Output score in range (-1, 1)

def state_to_tensor(state):
    """
    Convert a 3x3x3x3 Ultimate Tic-Tac-Toe board state into a 2x9x9 tensor for the neural network.
    """
    board_3x3x3x3 = state.board.copy()

    # Convert 3x3x3x3 nested board into 9x9
    board_9x9 = np.zeros((9, 9), dtype=np.float32)
    
    for meta_row in range(3):
        for meta_col in range(3):
            for local_row in range(3):
                for local_col in range(3):
                    global_row = meta_row * 3 + local_row
                    global_col = meta_col * 3 + local_col
                    board_9x9[global_row][global_col] = board_3x3x3x3[meta_row][meta_col][local_row][local_col]

    # Normalize values: 0 stays 0, AI (1) stays 1, Opponent (2) becomes -1
    board_9x9[board_9x9 == 2] = -1

    # Convert to PyTorch tensor and add batch dimension
    board_tensor = torch.tensor(board_9x9, dtype=torch.float32).unsqueeze(0)  # Shape: (1, 9, 9)
    
    turn_tensor = torch.full((1,9,9), 1 if state.fill_num==1 else -1, dtype=torch.float32)
    
    action_9x9 = np.zeros((9,9), dtype=np.float32)
    valid_actions = get_all_valid_actions(state)
    for meta_row, meta_col, local_row, local_col in valid_actions:
        global_row = meta_row * 3 + local_row
        global_col = meta_col * 3 + local_col
        action_9x9[global_row][global_col] = 1
    
    action_tensor = torch.tensor(action_9x9, dtype=torch.float32).unsqueeze(0)
    
    return torch.cat([turn_tensor,board_tensor,action_tensor],dim=0)

def evaluation(state, model):
    """
    Evaluates the board using the trained neural network.
    """
    if state.is_terminal(): #correct?
        return 2*state.terminal_utility()-1
    state_tensor = state_to_tensor(state).unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        return model(state_tensor).item()  # Get NN evaluation score
    
def minimax(model, state, depth, alpha, beta, maximizing=True):
    if depth == 0 or state.is_terminal():
        return evaluation(state, model), None
    
    best_action = None
    
    if maximizing:
        max_eval = -float("inf")
        for action in state.get_all_valid_actions():
            new_state = state.change_state(action)
            eval, _ = minimax(model, new_state, depth - 1, alpha, beta, False)
            if eval > max_eval:
                max_eval = eval
                best_action = action
            alpha = max(alpha, eval)
            if beta <= alpha:
                break
        return max_eval, best_action
    else:
        min_eval = float("inf")
        for action in state.get_all_valid_actions():
            new_state = state.change_state(action)
            eval, _ = minimax(model, new_state, depth - 1, alpha, beta, True)
            if eval < min_eval:
                min_eval = eval
                best_action = action
            beta = min(beta, eval)
            if beta <= alpha:
                break
        return min_eval, best_action


class StudentAgent:
    def __init__(self):
        """Instantiates your agent.
        """
        self.model=UTTTModel()
        self.model.load_state_dict(coeffs)

    def choose_action(self, state: State) -> Action:
        """Returns a valid action to be played on the board.
        Assuming that you are filling in the board with number 1.

        Parameters
        ---------------
        state: The board to make a move on.
        """
        best_score, best_move = minimax(self.model, state, depth=3, alpha=float("-inf"), beta=float("inf"), maximizing=True)
        return best_move
